# 2) Silver Layer
## Barcelona Urban Intelligence Assistant
### Advanced Data Processing and Analysis
The gold layer answers the questions, where bronze preserved the data, and silver standardised the data, so the gold layer transforms standardized data into decisions. 

Gold tables:
- Dim_district: “Which districts exist? What are their IDs?”
- Dim_date: “What day of the week/ month did this complaint arrive?”
- Fct_complaints_daily: “How many complaints per district per day?”
- Fct_resolution_times: “Which complaint types take longest by district and channel?”
- Fct_accidents_by_cause: “What causes accidents, where, and when?”
- Fct_airquality_monthly: “Is this district’s air above EU limits? In which months?”
- District_profiles: one row per district, combining all metrics for ML and advisory
- Regression_ready: One row per complaint, containing all features for linear regression
- Ev_advisory: “Should a citizen buy an EV in their district today?”


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import os

con = duckdb.connect("barcelona_urban.db")
print("Connected to barcelona_urban.db")


Connected to barcelona_urban.db


## Dimension Tables

In [51]:
# dim_district 
con.execute("""
CREATE OR REPLACE TABLE gold.dim_district AS
SELECT
    ROW_NUMBER() OVER (ORDER BY district) AS district_id,
    district AS district_name
FROM (
    SELECT DISTINCT district FROM silver.complaints  WHERE district != 'UNKNOWN'
    UNION
    SELECT DISTINCT district FROM silver.accidents   WHERE district != 'UNKNOWN'
    UNION
    SELECT DISTINCT district FROM silver.airquality  WHERE district != 'UNKNOWN'
) t
""")
print("gold.dim_district:")
display(con.execute("SELECT * FROM gold.dim_district ORDER BY district_id").df())


gold.dim_district:


,district_id,district_name
0,1,CIUTAT VELLA
1,2,EIXAMPLE
2,3,GRACIA
3,4,HORTA GUINARDO
4,5,LES CORTS
5,6,NOU BARRIS
6,7,SANT ANDREU
7,8,SANT MARTI
8,9,SANTS MONTJUIC
9,10,SARRIA SANT GERVASI


In [52]:
# dim_date 
con.execute("""
CREATE OR REPLACE TABLE gold.dim_date AS
SELECT DISTINCT
    CAST(STRFTIME(complaint_date, '%Y%m%d') AS INTEGER) AS date_id,
    complaint_date                                       AS full_date,
    YEAR(complaint_date)                                 AS year,
    MONTH(complaint_date)                                AS month,
    DAY(complaint_date)                                  AS day,
    DAYOFWEEK(complaint_date)                            AS weekday_num,
    CASE DAYOFWEEK(complaint_date)
        WHEN 1 THEN 'Monday'    WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday' WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'    WHEN 6 THEN 'Saturday'
        WHEN 0 THEN 'Sunday'
    END                                                  AS weekday_name
FROM silver.complaints
WHERE complaint_date IS NOT NULL
""")
n = con.execute("SELECT COUNT(*) FROM gold.dim_date").fetchone()[0]
print(f"gold.dim_date: {n:,} rows")
con.execute("SELECT * FROM gold.dim_date ORDER BY full_date LIMIT 5").df()


gold.dim_date: 575 rows


,date_id,full_date,year,month,day,weekday_num,weekday_name
0,20231025,2023-10-25,2023,10,25,3,Wednesday
1,20231214,2023-12-14,2023,12,14,4,Thursday
2,20240123,2024-01-23,2024,1,23,2,Tuesday
3,20240227,2024-02-27,2024,2,27,2,Tuesday
4,20240314,2024-03-14,2024,3,14,4,Thursday


In [53]:
# dim_complaint_type 
con.execute("""
CREATE OR REPLACE TABLE gold.dim_complaint_type AS
SELECT
    ROW_NUMBER() OVER (ORDER BY complaint_type, area) AS type_id,
    complaint_type,
    area,
    element
FROM (
    SELECT DISTINCT complaint_type, area, element
    FROM silver.complaints
    WHERE complaint_type != 'Unknown'
) t
""")
n = con.execute("SELECT COUNT(*) FROM gold.dim_complaint_type").fetchone()[0]
print(f"gold.dim_complaint_type: {n:,} rows")
con.execute("SELECT * FROM gold.dim_complaint_type ORDER BY complaint_type, area LIMIT 5").df()


gold.dim_complaint_type: 1,032 rows


,type_id,complaint_type,area,element
0,1,AGRAIMENT,Agraïments,Agraïments districtes
1,2,AGRAIMENT,Agraïments,Agraïments manteniment de l'espai urbà
2,3,AGRAIMENT,Agraïments,Agraïments atenció a la ciutadania
3,4,AGRAIMENT,Agraïments,Agraïments Serveis Socials
4,5,AGRAIMENT,Agraïments,Agraïments cultura


## Fact Tables

In [54]:
# fct_complaints_daily 
con.execute("""
CREATE OR REPLACE TABLE gold.fct_complaints_daily AS
SELECT
    complaint_date,
    district,
    COUNT(*)                                              AS complaint_count,
    AVG(resolution_days)                                  AS avg_resolution_days,
    MEDIAN(resolution_days)                               AS median_resolution_days,
    COUNT(CASE WHEN resolution_days IS NULL THEN 1 END)   AS unresolved_count
FROM silver.complaints
WHERE complaint_date IS NOT NULL
GROUP BY complaint_date, district
""")
n = con.execute("SELECT COUNT(*) FROM gold.fct_complaints_daily").fetchone()[0]
print(f"gold.fct_complaints_daily: {n:,} rows")
con.execute("SELECT * FROM gold.fct_complaints_daily ORDER BY district").df()



gold.fct_complaints_daily: 4,829 rows


,complaint_date,district,complaint_count,avg_resolution_days,median_resolution_days,unresolved_count
0,2025-10-10,CIUTAT VELLA,88,8.613636,5.0,0
1,2025-10-25,CIUTAT VELLA,36,7.388889,4.0,0
2,2025-10-29,CIUTAT VELLA,70,6.914286,2.0,0
3,2025-11-01,CIUTAT VELLA,32,9.093750,5.0,0
4,2025-10-18,CIUTAT VELLA,42,6.357143,3.0,0
...,...,...,...,...,...,...
4824,2025-06-10,UNKNOWN,370,11.662162,3.0,0
4825,2025-12-03,UNKNOWN,305,5.439344,2.0,0
4826,2025-07-01,UNKNOWN,396,11.237374,3.0,0
4827,2024-07-03,UNKNOWN,1,208.000000,208.0,0


In [55]:
# fct_resolution_times 
con.execute("""
CREATE OR REPLACE TABLE gold.fct_resolution_times AS
SELECT
    district,
    complaint_type,
    area,
    channel,
    COUNT(*)                      AS complaint_count,
    AVG(resolution_days)          AS avg_resolution_days,
    MEDIAN(resolution_days)       AS median_resolution_days,
    MIN(resolution_days)          AS min_days,
    MAX(resolution_days)          AS max_days,
    STDDEV(resolution_days)       AS stddev_days
FROM silver.complaints
WHERE resolution_days IS NOT NULL
  AND resolution_days >= 0
GROUP BY district, complaint_type, area, channel
""")
n = con.execute("SELECT COUNT(*) FROM gold.fct_resolution_times").fetchone()[0]
print(f"gold.fct_resolution_times: {n:,} rows")
con.execute("SELECT * FROM gold.fct_resolution_times ORDER BY avg_resolution_days DESC LIMIT 5").df()


gold.fct_resolution_times: 3,079 rows


,district,complaint_type,area,channel,complaint_count,avg_resolution_days,median_resolution_days,min_days,max_days,stddev_days
0,SANTS MONTJUIC,Incident,Urbanisme,RECLAMACIÓ INTERNA,1,352.0,352.0,352,352,NaN
1,CIUTAT VELLA,Complaint,Transports públics,RECLAMACIÓ INTERNA,1,231.0,231.0,231,231,NaN
2,GRACIA,Complaint,Cultura,TELÈFON,1,219.0,219.0,219,219,NaN
3,GRACIA,Incident,Transports públics,RECLAMACIÓ INTERNA,5,166.2,57.0,35,435,175.073413
4,EIXAMPLE,SUGGESTION,Transports públics,RECLAMACIÓ INTERNA,1,158.0,158.0,158,158,NaN


In [56]:
# fct_accidents_by_cause 
con.execute("""
CREATE OR REPLACE TABLE gold.fct_accidents_by_cause AS
SELECT
    district,
    accident_cause,
    year,
    month,
    COUNT(*)                AS accident_count,
    COUNT(DISTINCT accident_date) AS days_with_accidents
FROM silver.accidents
GROUP BY district, accident_cause, year, month
""")
n = con.execute("SELECT COUNT(*) FROM gold.fct_accidents_by_cause").fetchone()[0]
print(f"gold.fct_accidents_by_cause: {n:,} rows")
con.execute("SELECT * FROM gold.fct_accidents_by_cause ORDER BY accident_count DESC LIMIT 5").df()


gold.fct_accidents_by_cause: 1,484 rows


,district,accident_cause,year,month,accident_count,days_with_accidents
0,EIXAMPLE,Manca d'atenció a la conducció,2025,10,46,23
1,EIXAMPLE,Manca d'atenció a la conducció,2025,6,44,22
2,EIXAMPLE,Manca d'atenció a la conducció,2025,7,42,24
3,EIXAMPLE,Manca d'atenció a la conducció,2025,11,41,20
4,EIXAMPLE,Manca d'atenció a la conducció,2025,2,40,20


In [57]:
# fct_airquality_monthly 
con.execute("""
CREATE OR REPLACE TABLE gold.fct_airquality_monthly AS
SELECT
    district,
    pollutant_name,
    year,
    month,
    ROUND(monthly_avg_ugm3, 2) AS avg_ugm3,
    days_measured,
    -- EU threshold flags
    CASE WHEN pollutant_name = 'NO2'  AND monthly_avg_ugm3 > 40  THEN 1 ELSE 0 END AS exceeds_eu_no2,
    CASE WHEN pollutant_name = 'PM10' AND monthly_avg_ugm3 > 40  THEN 1 ELSE 0 END AS exceeds_eu_pm10
FROM silver.airquality
""")
n = con.execute("SELECT COUNT(*) FROM gold.fct_airquality_monthly").fetchone()[0]
print(f"gold.fct_airquality_monthly: {n:,} rows")
con.execute("SELECT * FROM gold.fct_airquality_monthly ORDER BY district, year, month LIMIT 5").df()

gold.fct_airquality_monthly: 143 rows


,district,pollutant_name,year,month,avg_ugm3,days_measured,exceeds_eu_no2,exceeds_eu_pm10
0,CIUTAT VELLA,NO2,2025,1,29.61,31,0,0
1,CIUTAT VELLA,NO2,2025,10,21.19,31,0,0
2,CIUTAT VELLA,NO2,2025,11,25.45,30,0,0
3,CIUTAT VELLA,NO2,2025,12,26.74,31,0,0
4,CIUTAT VELLA,NO2,2025,2,30.00,28,0,0


## gold.district_profiles: he Core ML & RAG Table
One row per district. Contains all features + three SDG-aligned scores.
This is the input for K-Means clustering and what the RAG assistant retrieves.

In [58]:
con.execute("""
CREATE OR REPLACE TABLE gold.district_profiles AS

WITH
-- Total complaints + avg resolution per district
complaints_agg AS (
    SELECT
        district,
        COUNT(*)                         AS total_complaints,
        AVG(resolution_days)             AS avg_resolution_days,
        MEDIAN(resolution_days)          AS median_resolution_days,
        COUNT(CASE WHEN resolution_days > 30 THEN 1 END) AS complaints_over_30_days
    FROM silver.complaints
    WHERE district != 'UNKNOWN'
    GROUP BY district
),

-- Total accidents + most common cause per district
accidents_agg AS (
    SELECT
        district,
        COUNT(*)                         AS total_accidents,
        -- Most frequent cause
        MODE(accident_cause)             AS dominant_accident_cause,
        -- Proportion of distraction + speeding (Vision Zero risk indicators)
        ROUND(100.0 * COUNT(CASE WHEN accident_cause IN
            ('Driver distraction','Inappropriate speed','Drink driving','Drug impairment')
            THEN 1 END) / COUNT(*), 1)   AS pct_high_risk_causes
    FROM silver.accidents
    WHERE district != 'UNKNOWN'
    GROUP BY district
),

-- Annual NO2 and PM10 per district (average across all months measured)
airquality_agg AS (
    SELECT
        district,
        ROUND(AVG(CASE WHEN pollutant_name = 'NO2'  THEN avg_ugm3 END), 2) AS annual_avg_no2,
        ROUND(AVG(CASE WHEN pollutant_name = 'PM10' THEN avg_ugm3 END), 2) AS annual_avg_pm10,
        SUM(CASE WHEN pollutant_name = 'NO2'  AND exceeds_eu_no2  = 1 THEN 1 ELSE 0 END) AS months_exceeding_no2_limit,
        SUM(CASE WHEN pollutant_name = 'PM10' AND exceeds_eu_pm10 = 1 THEN 1 ELSE 0 END) AS months_exceeding_pm10_limit
    FROM gold.fct_airquality_monthly
    WHERE district != 'UNKNOWN'
    GROUP BY district
),

-- City-wide PM10 mean: used to impute districts with NO2 station but no PM10 sensor.
-- NO2 is NOT imputed the same way: Nou Barris and Sant Andreu have NO station at all,
-- so their NO2 is genuinely unknown and stays NULL (imputing would be misleading).
-- PM10 gaps are a different case: a district may have an NO2 station but no PM10 sensor,
-- and the city mean is a defensible estimate for an advisory tool (flagged in catalogue).
pm10_city_mean AS (
    SELECT ROUND(AVG(avg_ugm3), 2) AS city_mean_pm10
    FROM gold.fct_airquality_monthly
    WHERE pollutant_name = 'PM10'
),

-- EV charging per district
ev_agg AS (
    SELECT district, ev_charging_points
    FROM silver.ev_charging
    WHERE district != 'UNKNOWN'
),

-- Motorization index for private cars (Turismes) per district
vehicles_agg AS (
    SELECT district,
           ROUND(AVG(avg_motorization_index), 1) AS car_motorization_index
    FROM silver.vehicles
    WHERE vehicle_type = 'Turismes'
      AND district != 'UNKNOWN'
    GROUP BY district
)

-- ── FINAL JOIN 
SELECT
    d.district_id,
    d.district_name                                       AS district,

    -- Complaints features
    COALESCE(c.total_complaints, 0)                       AS total_complaints,
    ROUND(COALESCE(c.avg_resolution_days, 0), 1)          AS avg_resolution_days,
    ROUND(COALESCE(c.median_resolution_days, 0), 1)       AS median_resolution_days,
    COALESCE(c.complaints_over_30_days, 0)                AS complaints_over_30_days,

    -- Accident features
    COALESCE(a.total_accidents, 0)                        AS total_accidents,
    COALESCE(a.dominant_accident_cause, 'Unknown')        AS dominant_cause,
    COALESCE(a.pct_high_risk_causes, 0)                   AS pct_high_risk_causes,

    -- Air quality features
    -- NO2: kept NULL for Nou Barris + Sant Andreu (no monitoring station — not imputed).
    aq.annual_avg_no2,
    -- PM10: impute with Barcelona city mean for districts that have an NO2 station
    --       but no PM10 sensor (different from the no-station-at-all case for NO2).
    --       Imputed value flagged in data catalogue as DQ-07 Completeness.
    COALESCE(
        aq.annual_avg_pm10,
        (SELECT city_mean_pm10 FROM pm10_city_mean)
    )                                                      AS annual_avg_pm10,
    COALESCE(aq.months_exceeding_no2_limit, 0)            AS months_exceeding_no2,
    COALESCE(aq.months_exceeding_pm10_limit, 0)           AS months_exceeding_pm10,

    -- EV infrastructure
    COALESCE(ev.ev_charging_points, 0)                    AS ev_charging_points,

    -- Vehicle ownership
    -- NOTE: keep NULL if data is missing — do NOT default to 0.
    -- Zero would be mathematically wrong (no city district has zero cars).
    -- Districts with NULL here are Gracia, Les Corts, Sarria (source data gap).
    v.car_motorization_index                              AS car_motorization_index,

    -- ── SDG SCORES 
    -- SDG 3 / Vision Zero: accidents per district (lower = better, 0–10 scale)
    ROUND(10.0 * COALESCE(a.total_accidents, 0) /
        NULLIF(MAX(a.total_accidents) OVER (), 0), 2)     AS vision_zero_risk_score,

    -- SDG 13 / Paris: 1 if NO2 below EU limit (40 µg/m3), 0 if exceeding, NULL if no station
    CASE
        WHEN aq.annual_avg_no2 IS NULL THEN NULL       -- no monitoring station
        WHEN aq.annual_avg_no2 <= 40   THEN 1
        ELSE 0
    END                                                   AS paris_no2_compliant,
    CASE
        WHEN aq.annual_avg_no2 IS NULL THEN NULL
        WHEN aq.annual_avg_no2 <= 10   THEN 1
        ELSE 0
    END                                                   AS who_no2_compliant,

    -- SDG 11 / EU 2035: EV transition risk score
    -- Formula: car_motorization_index / ev_charging_points
    -- Higher = more cars relative to chargers = higher risk from 2035 ICE ban
    -- NULL when car data is missing (do NOT use 0 — it gives false "low risk" signal)
    CASE
        WHEN v.car_motorization_index IS NULL THEN NULL
        WHEN COALESCE(ev.ev_charging_points, 0) = 0    THEN 999.0  -- no chargers at all
        ELSE ROUND(v.car_motorization_index / ev.ev_charging_points, 2)
    END                                                   AS ev_transition_gap_score

FROM gold.dim_district d
LEFT JOIN complaints_agg  c  ON d.district_name = c.district
LEFT JOIN accidents_agg   a  ON d.district_name = a.district
LEFT JOIN airquality_agg  aq ON d.district_name = aq.district
LEFT JOIN ev_agg          ev ON d.district_name = ev.district
LEFT JOIN vehicles_agg    v  ON d.district_name = v.district
ORDER BY d.district_id
""")

print("gold.district_profiles — THE CORE ML TABLE:")
display(con.execute("""
    SELECT district,
           total_complaints, total_accidents,
           annual_avg_no2, ev_charging_points, car_motorization_index,
           vision_zero_risk_score, paris_no2_compliant, ev_transition_gap_score
    FROM gold.district_profiles
    ORDER BY vision_zero_risk_score DESC
""").df())
con.execute("SELECT * FROM gold.district_profiles ORDER BY vision_zero_risk_score DESC LIMIT 5").df()


gold.district_profiles — THE CORE ML TABLE:


,district,total_complaints,total_accidents,annual_avg_no2,ev_charging_points,car_motorization_index,vision_zero_risk_score,paris_no2_compliant,ev_transition_gap_score
0,EIXAMPLE,34608,2009,29.38,23,267.1,10.00,1,11.61
1,SANT MARTI,32321,1097,22.32,20,302.1,5.46,1,15.11
2,SANTS MONTJUIC,21985,963,17.98,12,233.8,4.79,1,19.48
3,SARRIA SANT GERVASI,12941,828,7.45,19,271.9,4.12,1,14.31
4,HORTA GUINARDO,20151,630,18.65,10,279.4,3.14,1,27.94
5,LES CORTS,7056,624,15.45,13,271.9,3.11,1,20.92
6,SANT ANDREU,16358,587,NaN,13,296.7,2.92,<NA>,22.82
7,NOU BARRIS,17067,487,NaN,6,252.3,2.42,<NA>,42.05
8,GRACIA,15007,428,25.47,9,271.9,2.13,1,30.21
9,CIUTAT VELLA,19874,419,22.44,12,271.9,2.09,1,22.66


,district_id,district,total_complaints,avg_resolution_days,median_resolution_days,complaints_over_30_days,total_accidents,dominant_cause,pct_high_risk_causes,annual_avg_no2,annual_avg_pm10,months_exceeding_no2,months_exceeding_pm10,ev_charging_points,car_motorization_index,vision_zero_risk_score,paris_no2_compliant,who_no2_compliant,ev_transition_gap_score
0,2,EIXAMPLE,34608,7.4,3.0,1651,2009,Manca d'atenció a la conducció,0.0,29.38,22.58,0.0,0.0,23,267.1,10.00,1,0,11.61
1,8,SANT MARTI,32321,6.8,3.0,1077,1097,Manca d'atenció a la conducció,0.0,22.32,21.79,0.0,0.0,20,302.1,5.46,1,0,15.11
2,9,SANTS MONTJUIC,21985,11.2,5.0,1388,963,Manca d'atenció a la conducció,0.0,17.98,19.42,0.0,0.0,12,233.8,4.79,1,0,19.48
3,10,SARRIA SANT GERVASI,12941,10.1,5.0,726,828,Manca d'atenció a la conducció,0.0,7.45,17.76,0.0,0.0,19,271.9,4.12,1,1,14.31
4,4,HORTA GUINARDO,20151,9.8,4.0,1219,630,Manca d'atenció a la conducció,0.0,18.65,17.49,0.0,0.0,10,279.4,3.14,1,0,27.94


In [59]:
# district_profiles full table (10 rows)
print("gold.district_profiles, full table (10 rows)")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
display(con.execute("""
    SELECT district, total_complaints, avg_resolution_days,
           total_accidents, annual_avg_no2, annual_avg_pm10,
           ev_charging_points, car_motorization_index,
           vision_zero_risk_score, paris_no2_compliant, ev_transition_gap_score
    FROM gold.district_profiles
    ORDER BY vision_zero_risk_score DESC
""").df())


gold.district_profiles, full table (10 rows)


,district,total_complaints,avg_resolution_days,total_accidents,annual_avg_no2,annual_avg_pm10,ev_charging_points,car_motorization_index,vision_zero_risk_score,paris_no2_compliant,ev_transition_gap_score
0,EIXAMPLE,34608,7.4,2009,29.38,22.58,23,267.1,10.00,1,11.61
1,SANT MARTI,32321,6.8,1097,22.32,21.79,20,302.1,5.46,1,15.11
2,SANTS MONTJUIC,21985,11.2,963,17.98,19.42,12,233.8,4.79,1,19.48
3,SARRIA SANT GERVASI,12941,10.1,828,7.45,17.76,19,271.9,4.12,1,14.31
4,HORTA GUINARDO,20151,9.8,630,18.65,17.49,10,279.4,3.14,1,27.94
5,LES CORTS,7056,10.3,624,15.45,17.47,13,271.9,3.11,1,20.92
6,SANT ANDREU,16358,8.3,587,NaN,19.42,13,296.7,2.92,<NA>,22.82
7,NOU BARRIS,17067,9.6,487,NaN,19.42,6,252.3,2.42,<NA>,42.05
8,GRACIA,15007,8.5,428,25.47,19.42,9,271.9,2.13,1,30.21
9,CIUTAT VELLA,19874,8.7,419,22.44,19.42,12,271.9,2.09,1,22.66


## Seeing if no district is left behind

In [60]:
print("Gold layer completeness check:\n")

CANONICAL_10 = {
    "CIUTAT VELLA", "EIXAMPLE", "SANTS MONTJUIC", "LES CORTS",
    "SARRIA SANT GERVASI", "GRACIA", "HORTA GUINARDO",
    "NOU BARRIS", "SANT ANDREU", "SANT MARTI"
}

gold = con.execute("SELECT * FROM gold.district_profiles").df()
passed = True

# Row count 
if len(gold) == 10:
    print(f"Row count: {len(gold)} districts (expected 10)")
else:
    print(f"Row count: {len(gold)} — expected 10")
    in_gold     = set(gold["district"])
    missing     = CANONICAL_10 - in_gold
    unexpected  = in_gold - CANONICAL_10
    if missing:    print(f"   Missing from Gold:    {sorted(missing)}")
    if unexpected: print(f"   Unexpected in Gold:   {sorted(unexpected)}")
    passed = False

# PM10: should have a value for every district after imputation 
pm10_null = gold.loc[gold["annual_avg_pm10"].isna(), "district"].tolist()
if not pm10_null:
    print("annual_avg_pm10: all 10 districts have a value (real or city-mean imputed)")
else:
    print(f" annual_avg_pm10 still NULL for: {pm10_null}")
    print("   Imputation CTE found no PM10 data at all — check fct_airquality_monthly")
    passed = False

# NO2: NULL only for the two known no-station districts 
no2_null = set(gold.loc[gold["annual_avg_no2"].isna(), "district"].tolist())
EXPECTED_NO2_NULL = {"NOU BARRIS", "SANT ANDREU"}
unexpected_no2_null = no2_null - EXPECTED_NO2_NULL
missing_no2_null    = EXPECTED_NO2_NULL - no2_null
if not unexpected_no2_null and not missing_no2_null:
    print(f"annual_avg_no2: NULL only for {sorted(EXPECTED_NO2_NULL)} (expected — no station)")
else:
    if unexpected_no2_null:
        print(f" Unexpected NULL NO2 for: {sorted(unexpected_no2_null)}")
        passed = False
    if missing_no2_null:
        print(f" NO2 now populated for: {sorted(missing_no2_null)} (previously NULL — check if correct)")
# car_motorization_index: all 10 districts (imputed with city avg in Silver) 
null_motor = gold.loc[gold["car_motorization_index"].isna(), "district"].tolist()
if not null_motor:
    print("car_motorization_index: all 10 districts populated")
else:
    print(f"car_motorization_index NULL for: {null_motor}")
    print("   Silver vehicle imputation did not cover these districts.")
    print("   Check that CANONICAL_DISTRICTS in Silver matches the spellings here.")
    passed = False

# ev_transition_gap_score: no false 0.0 (COALESCE bug guard) 
false_zero = gold.loc[
    (gold["ev_transition_gap_score"] == 0.0) &
    (gold["car_motorization_index"].notna()),
    "district"
].tolist()
if not false_zero:
    print(" ev_transition_gap_score: no false 0.0 values")
else:
    print(f" ev_transition_gap_score = 0.0 for: {false_zero}")
    print("   This means car_motorization_index was COALESCED to 0 somewhere.")
    print("   Search for COALESCE(v.car_motorization_index, 0) and remove it.")
    passed = False

# vision_zero_risk_score: 0–10 range 
bad_vz = gold.loc[(gold["vision_zero_risk_score"] < 0) | (gold["vision_zero_risk_score"] > 10)]
if len(bad_vz) == 0:
    print(" vision_zero_risk_score: all values in [0, 10]")
else:
    print(f" Out-of-range vision_zero_risk_score: {bad_vz[['district','vision_zero_risk_score']].values}")
    passed = False

print(f"\n{'='*50}")
print(" ALL ASSERTIONS PASSED" if passed else " SOME ASSERTIONS FAILED — review above before running ML")


Gold layer completeness check:

Row count: 10 districts (expected 10)
annual_avg_pm10: all 10 districts have a value (real or city-mean imputed)
annual_avg_no2: NULL only for ['NOU BARRIS', 'SANT ANDREU'] (expected — no station)
car_motorization_index: all 10 districts populated
 ev_transition_gap_score: no false 0.0 values
 vision_zero_risk_score: all values in [0, 10]

 ALL ASSERTIONS PASSED


## gold.regression_ready 
One row per complaint, all features as columns, target = resolution_days.

In [61]:
con.execute("""
CREATE OR REPLACE TABLE gold.regression_ready AS
SELECT
    c.resolution_days                   AS target_resolution_days,
    d.district_id,
    c.district,
    c.complaint_year                    AS year,
    c.complaint_month                   AS month,
    DAYOFWEEK(c.complaint_date)         AS weekday,
    c.channel,
    c.complaint_type,
    c.area,
    -- Enrich with district-level features from district_profiles
    COALESCE(p.total_accidents, 0)      AS district_accidents,
    COALESCE(p.annual_avg_no2, 0)       AS district_no2,
    COALESCE(p.ev_charging_points, 0)   AS district_ev_points,
    COALESCE(p.car_motorization_index,0)AS district_motorization
FROM silver.complaints c
LEFT JOIN gold.dim_district   d ON c.district = d.district_name
LEFT JOIN gold.district_profiles p ON c.district = p.district
WHERE c.resolution_days IS NOT NULL
  AND c.resolution_days >= 0
  AND c.resolution_days <= 365        -- remove extreme outliers
""")
n = con.execute("SELECT COUNT(*) FROM gold.regression_ready").fetchone()[0]
print(f"gold.regression_ready: {n:,} rows")
con.execute("SELECT * FROM gold.regression_ready ORDER BY target_resolution_days DESC LIMIT 5").df()


gold.regression_ready: 287,272 rows


,target_resolution_days,district_id,district,year,month,weekday,channel,complaint_type,area,district_accidents,district_no2,district_ev_points,district_motorization
0,365,9,SANTS MONTJUIC,2024,7,2,MÒBIL,Incident,Transports públics,963,17.98,12,233.8
1,356,5,LES CORTS,2024,8,4,RECLAMACIÓ INTERNA,Incident,Manteniment de l'espai urbà,624,15.45,13,271.9
2,352,9,SANTS MONTJUIC,2024,7,6,RECLAMACIÓ INTERNA,Incident,Urbanisme,963,17.98,12,233.8
3,351,5,LES CORTS,2024,8,2,RECLAMACIÓ INTERNA,Incident,Manteniment de l'espai urbà,624,15.45,13,271.9
4,342,<NA>,UNKNOWN,2024,3,4,TELÈFON,Incident,Informació tràmits i atenció ciutadana,0,0.00,0,0.0


In [ ]:
# regression_ready: summary statistics (too many rows to show fully)
print(f"gold.regression_ready summary statistics")
display(con.execute("""
    SELECT complaint_type, district, channel,
           ROUND(AVG(target_resolution_days),1) AS avg_days,
           COUNT(*) AS n_complaints
    FROM gold.regression_ready
    GROUP BY complaint_type, district, channel
    ORDER BY avg_days DESC
    LIMIT 15
""").df())

gold.regression_ready summary statistics


,complaint_type,district,channel,avg_days,n_complaints
0,COMPLAINT,UNKNOWN,CORREU ELECTRÒNIC,114.0,1
1,Enquiry,CIUTAT VELLA,RECLAMACIÓ INTERNA,96.0,1
2,QUERY,NOU BARRIS,RECLAMACIÓ INTERNA,82.3,3
3,COMPLAINT,CIUTAT VELLA,CORREU ELECTRÒNIC,74.0,1
4,SERVICE REQUEST,SANTS MONTJUIC,RECLAMACIÓ INTERNA,72.7,3
5,Suggestion,NOU BARRIS,CORREU ELECTRÒNIC,70.0,1
6,Suggestion,LES CORTS,PROXIMITAT GUB,62.0,1
7,SUGGESTION,SANT ANDREU,RECLAMACIÓ INTERNA,60.6,25
8,Suggestion,EIXAMPLE,ALTRES SUPORTS,58.0,2
9,COMPLAINT,CIUTAT VELLA,RECLAMACIÓ INTERNA,55.9,65


## Data Quality Checks Before Advisory
Run these checks every time before generating the advisory table.
They catch NULL propagation bugs and missing join data early.

In [63]:
print("Gold layer data quality checks\n")

checks = con.execute("""
SELECT
    district,
    total_complaints,
    total_accidents,
    annual_avg_no2,
    ev_charging_points,
    car_motorization_index,
    ev_transition_gap_score,

    -- Flag each potential issue
    CASE WHEN car_motorization_index IS NULL     THEN '❌ MISSING' ELSE '✅' END AS vehicles_ok,
    CASE WHEN annual_avg_no2       IS NULL       THEN '⚪ NO STATION' ELSE '✅' END AS airquality_ok,
    CASE WHEN ev_charging_points   = 0           THEN '⚠️  NO CHARGERS' ELSE '✅' END AS ev_ok,
    CASE WHEN ev_transition_gap_score = 0.0
          AND car_motorization_index > 0         THEN '❌ DIVIDE ERROR' ELSE '✅' END AS gap_score_ok
FROM gold.district_profiles
ORDER BY district
""").df()

display(checks[["district","vehicles_ok","airquality_ok","ev_ok","gap_score_ok",
                "car_motorization_index","ev_charging_points","ev_transition_gap_score"]])

# Summary
missing_v  = checks["vehicles_ok"].str.contains("MISSING").sum()
no_station = checks["airquality_ok"].str.contains("NO STATION").sum()
no_charger = checks["ev_ok"].str.contains("NO CHARGERS").sum()
bad_gap    = checks["gap_score_ok"].str.contains("ERROR").sum()

print(f"\n  Districts missing vehicle data:     {missing_v} (source portal gap — imputed in Silver)")
print(f"  Districts without air quality station: {no_station} (Nou Barris, Sant Andreu — no station exists)")
print(f"  Districts with no EV chargers:       {no_charger}")
print(f"  Districts with suspicious gap score: {bad_gap}")
if bad_gap == 0 and missing_v == 0:
    print("\n✅ All checks passed — safe to generate advisory table.")
else:
    print("\n⚠️  Some issues found — review above before trusting the advisory output.")


Gold layer data quality checks



,district,vehicles_ok,airquality_ok,ev_ok,gap_score_ok,car_motorization_index,ev_charging_points,ev_transition_gap_score
0,CIUTAT VELLA,✅,✅,✅,✅,271.9,12,22.66
1,EIXAMPLE,✅,✅,✅,✅,267.1,23,11.61
2,GRACIA,✅,✅,✅,✅,271.9,9,30.21
3,HORTA GUINARDO,✅,✅,✅,✅,279.4,10,27.94
4,LES CORTS,✅,✅,✅,✅,271.9,13,20.92
5,NOU BARRIS,✅,⚪ NO STATION,✅,✅,252.3,6,42.05
6,SANT ANDREU,✅,⚪ NO STATION,✅,✅,296.7,13,22.82
7,SANT MARTI,✅,✅,✅,✅,302.1,20,15.11
8,SANTS MONTJUIC,✅,✅,✅,✅,233.8,12,19.48
9,SARRIA SANT GERVASI,✅,✅,✅,✅,271.9,19,14.31



  Districts missing vehicle data:     0 (source portal gap — imputed in Silver)
  Districts without air quality station: 2 (Nou Barris, Sant Andreu — no station exists)
  Districts with no EV chargers:       0
  Districts with suspicious gap score: 0

✅ All checks passed — safe to generate advisory table.


## Citizen EV advisory + charger placement ranking
Two practical outputs that directly answer the research questions:
1. **Should I buy an electric car?**:  per-district yes/no advice grounded in charging density and air quality
2. **Where should new chargers be built?:  districts ranked by urgency (most cars, fewest chargers first)

In [64]:
advisory = con.execute("""
SELECT
    district,
    total_accidents,
    ROUND(annual_avg_no2, 1)              AS avg_no2_ugm3,
    ev_charging_points,
    ROUND(car_motorization_index, 0)      AS cars_per_1000_people,
    ROUND(ev_transition_gap_score, 1)     AS cars_per_charger,
    vision_zero_risk_score,
    paris_no2_compliant,

    -- ── CITIZEN EV ADVICE
    CASE
        -- Guard: missing vehicle data → cannot give reliable advice
        WHEN car_motorization_index IS NULL
            THEN '⚠️  Vehicle data unavailable for your district. Check the city open data portal for local estimates.'
        -- No public chargers at all in this district
        WHEN ev_charging_points = 0 OR ev_transition_gap_score = 999.0
            THEN '❌ Not yet — no public chargers in your district. Wait for infrastructure rollout.'
        -- Very high gap: many cars, very few chargers (>25 cars per charger proxy)
        WHEN ev_transition_gap_score > 25
            THEN '⚠️  Possible but risky : few chargers relative to car density. Check your specific street before deciding.'
        -- High pollution + adequate charging → strongest case for EV
        WHEN annual_avg_no2 > 35 AND ev_transition_gap_score <= 25
            THEN '✅ Yes — your district has high air pollution AND reasonable charging coverage. Going electric has real, local health impact.'
        -- Good charging density
        WHEN ev_transition_gap_score <= 15
            THEN '✅ Yes — good charging infrastructure relative to car density. Electric is practical here.'
        -- No air quality data but moderate charging
        WHEN annual_avg_no2 IS NULL AND ev_transition_gap_score <= 25
            THEN '🔄 Likely yes — charging coverage is reasonable but no local air quality data to confirm pollution benefit.'
        ELSE
            '🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.'
    END AS ev_buy_advice,

    -- ── CHARGER PLACEMENT PRIORITY ────────────────────────────────────────
    -- High car ownership + few chargers = most urgent for new infrastructure
    RANK() OVER (
        ORDER BY (car_motorization_index / NULLIF(ev_charging_points + 1, 0)) DESC
    ) AS new_charger_priority

FROM gold.district_profiles
ORDER BY new_charger_priority
""").df()


print("CITIZEN ADVISORY: Should I buy an electric car?")

display(advisory[["district","avg_no2_ugm3","ev_charging_points",
                   "cars_per_charger","ev_buy_advice"]])

print("INFRASTRUCTURE PRIORITY: Where should new chargers be built?")
print("Rank 1 = most urgent need")
display(advisory[["new_charger_priority","district",
                  "cars_per_1000_people","ev_charging_points","cars_per_charger"]])


CITIZEN ADVISORY: Should I buy an electric car?


,district,avg_no2_ugm3,ev_charging_points,cars_per_charger,ev_buy_advice
0,NOU BARRIS,NaN,6,42.1,⚠️ Possible but risky : few chargers relative to car density. Check your specific street before deciding.
1,GRACIA,25.5,9,30.2,⚠️ Possible but risky : few chargers relative to car density. Check your specific street before deciding.
2,HORTA GUINARDO,18.7,10,27.9,⚠️ Possible but risky : few chargers relative to car density. Check your specific street before deciding.
3,SANT ANDREU,NaN,13,22.8,🔄 Likely yes — charging coverage is reasonable but no local air quality data to confirm pollution benefit.
4,CIUTAT VELLA,22.4,12,22.7,🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.
5,LES CORTS,15.5,13,20.9,🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.
6,SANTS MONTJUIC,18.0,12,19.5,🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.
7,SANT MARTI,22.3,20,15.1,🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.
8,SARRIA SANT GERVASI,7.5,19,14.3,✅ Yes — good charging infrastructure relative to car density. Electric is practical here.
9,EIXAMPLE,29.4,23,11.6,✅ Yes — good charging infrastructure relative to car density. Electric is practical here.


INFRASTRUCTURE PRIORITY: Where should new chargers be built?
Rank 1 = most urgent need


,new_charger_priority,district,cars_per_1000_people,ev_charging_points,cars_per_charger
0,1,NOU BARRIS,252.0,6,42.1
1,2,GRACIA,272.0,9,30.2
2,3,HORTA GUINARDO,279.0,10,27.9
3,4,SANT ANDREU,297.0,13,22.8
4,5,CIUTAT VELLA,272.0,12,22.7
5,6,LES CORTS,272.0,13,20.9
6,7,SANTS MONTJUIC,234.0,12,19.5
7,8,SANT MARTI,302.0,20,15.1
8,9,SARRIA SANT GERVASI,272.0,19,14.3
9,10,EIXAMPLE,267.0,23,11.6


In [65]:
pd.set_option('display.max_colwidth', None)
display(con.execute("""
    SELECT district, avg_no2_ugm3, ev_charging_points,
           cars_per_1000_people, cars_per_charger,
           ev_buy_advice, new_charger_priority
    FROM gold.ev_advisory
    ORDER BY new_charger_priority
""").df())


,district,avg_no2_ugm3,ev_charging_points,cars_per_1000_people,cars_per_charger,ev_buy_advice,new_charger_priority
0,NOU BARRIS,NaN,6,252.0,42.1,⚠️ Possible but risky — very few chargers relative to car density. Check your specific street before deciding.,1
1,GRACIA,25.5,9,272.0,30.2,⚠️ Possible but risky — very few chargers relative to car density. Check your specific street before deciding.,2
2,HORTA GUINARDO,18.7,10,279.0,27.9,⚠️ Possible but risky — very few chargers relative to car density. Check your specific street before deciding.,3
3,SANT ANDREU,NaN,13,297.0,22.8,🔄 Likely yes — charging coverage is reasonable but no local air quality data to confirm pollution benefit.,4
4,CIUTAT VELLA,22.4,12,272.0,22.7,🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.,5
5,LES CORTS,15.5,13,272.0,20.9,🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.,6
6,SANTS MONTJUIC,18.0,12,234.0,19.5,🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.,7
7,SANT MARTI,22.3,20,302.0,15.1,🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.,8
8,SARRIA SANT GERVASI,7.5,19,272.0,14.3,✅ Yes — good charging infrastructure relative to car density. Electric is practical here.,9
9,EIXAMPLE,29.4,23,267.0,11.6,✅ Yes — good charging infrastructure relative to car density. Electric is practical here.,10


In [66]:
rows = con.execute("""
    SELECT district, ev_buy_advice, new_charger_priority
    FROM gold.ev_advisory
    ORDER BY new_charger_priority
""").df()

for _, row in rows.iterrows():
    print(f"#{int(row['new_charger_priority'])} {row['district']}")
    print(f"   {row['ev_buy_advice']}")
    print()


#1 NOU BARRIS
   ⚠️  Possible but risky — very few chargers relative to car density. Check your specific street before deciding.

#2 GRACIA
   ⚠️  Possible but risky — very few chargers relative to car density. Check your specific street before deciding.

#3 HORTA GUINARDO
   ⚠️  Possible but risky — very few chargers relative to car density. Check your specific street before deciding.

#4 SANT ANDREU
   🔄 Likely yes — charging coverage is reasonable but no local air quality data to confirm pollution benefit.

#5 CIUTAT VELLA
   🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.

#6 LES CORTS
   🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.

#7 SANTS MONTJUIC
   🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.

#8 SANT MARTI
   🔄 Depends — charging exists but coverage is moderate. Verify availability near your home address.

#9 SARRIA SANT 

In [67]:
# Save advisory table to Gold so the RAG assistant can retrieve it
con.execute("CREATE OR REPLACE TABLE gold.ev_advisory AS SELECT * FROM advisory")
n = con.execute("SELECT COUNT(*) FROM gold.ev_advisory").fetchone()[0]
print(f"gold.ev_advisory: {n} district rows saved")
print("RAG assistant will use this table to answer citizen questions like:")
print("  'Should I buy an EV if I live in Gracia?'")
print("  'Which district most urgently needs new charging points?'")


gold.ev_advisory: 10 district rows saved
RAG assistant will use this table to answer citizen questions like:
  'Should I buy an EV if I live in Gracia?'
  'Which district most urgently needs new charging points?'


## Final Verification

In [71]:
print("=" * 60)
print("Gold layer complete: full database summary:")
print("=" * 60)

all_tables = con.execute("SHOW ALL TABLES").df()
for _, row in all_tables.iterrows():
    schema, name = row["schema"], row["name"]
    count = con.execute(f"SELECT COUNT(*) FROM {schema}.{name}").fetchone()[0]
    print(f"  {schema:6s}.{name:35s}: {count:>8,} rows")

print("\nSDG compliance:")
display(con.execute("""
    SELECT
        district,
        total_accidents,
        annual_avg_no2,
        ev_charging_points,
        car_motorization_index,
        CASE WHEN vision_zero_risk_score >= 7 THEN '🔴 HIGH RISK'
             WHEN vision_zero_risk_score >= 4 THEN '🟡 MODERATE'
             ELSE '🟢 LOW RISK' END                    AS vision_zero_status,
        CASE
            WHEN paris_no2_compliant IS NULL THEN '⚪ NO STATION DATA'
            WHEN paris_no2_compliant = 1     THEN '🟢 COMPLIANT'
            ELSE                                   '🔴 EXCEEDS EU LIMIT'
        END                                                AS paris_status,
        CASE WHEN ev_transition_gap_score IS NULL
              OR ev_transition_gap_score > 20 THEN '🔴 HIGH GAP'
             WHEN ev_transition_gap_score > 10 THEN '🟡 MODERATE GAP'
             ELSE '🟢 READY' END                       AS ev_2035_readiness
    FROM gold.district_profiles
    ORDER BY vision_zero_risk_score DESC NULLS LAST
""").df())

print("\nDatabase path:", os.path.abspath("barcelona_urban.db"))
print('  con = duckdb.connect("barcelona_urban.db")')
print('  con.execute("SELECT * FROM gold.district_profiles").df()')


Gold layer complete: full database summary:


ConnectionException: Connection Error: Connection already closed!

In [69]:
# close the connection when done
con.close()
print("Connection closed.")


Connection closed.


In [72]:
# close the connection when done 
con.close()
print(" All done. Connection closed.")


 All done. Connection closed.
